Create a SparkSession and read employees.csv into a DataFrame. Print the schema and show the first 5 rows.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('read').getOrCreate()

df = (
    spark.read
         .option('header', 'true')
         .option('inferSchema', 'true')   # fixed typo
         .csv('data/question_01.csv')
)
# print(df.dtypes)
df.printSchema()   # no print() wrapper
df.show(5)         # not head()

```text

show() is an action — triggers actual computation. 
printSchema() just reads the plan metadata, no Spark job is launched.

```

Q2. Filter employees earning above 60,000 and select only name and salary.

In [ ]:

df.filter(df['salary'] > 60000).select('name','salary').show()

''' Alternate methods
# 1. Using col() — most common in production code
from pyspark.sql.functions import col
df.filter(col('salary') > 60000).select('name', 'salary').show()

# 2. SQL-style string expression
df.filter('salary > 60000').select('name', 'salary').show()

# 3. where() — alias for filter(), same thing
df.where(df.salary > 60000).select('name', 'salary').show()



'''

+----------------+------+
|            name|salary|
+----------------+------+
|      Aarav Shah| 72000|
|      Sneha Iyer| 81000|
|     Karan Patel| 95000|
|      Neha Gupta| 88000|
|     Rohan Joshi| 61000|
|    Ananya Singh| 76000|
|      Vikram Rao|102000|
|    Arjun Pillai| 91000|
|   Kavitha Rajan| 83000|
|  Tejas Malhotra| 63000|
|     Nikhil Bose| 78000|
|    Suresh Reddy| 67000|
|     Deepak Nair| 79000|
|     Riya Kapoor| 85000|
|      Arun Kumar| 99000|
|   Ganesh Murthy| 70000|
|   Ishaan Thakur| 87000|
|    Jyoti Pandey| 74000|
|  Manish Agarwal| 65000|
|Nalini Venkatesh| 93000|
+----------------+------+
only showing top 20 rows


```text
Interview trap to know:

Why does .select() come after .filter() and not before?

Technically both orders work, but filter first → select second is correct because you reduce rows before reducing columns — less data shuffled. Interviewers love this follow-up.
```

Q3. Count total employees per department using groupBy + count, ordered from largest to smallest.



In [ ]:
df.show()

+------+----------------+-----------+------+---------+
|emp_id|            name|       dept|salary|join_year|
+------+----------------+-----------+------+---------+
|     1|      Aarav Shah|Engineering| 72000|     2019|
|     2|     Priya Menon|         HR| 45000|     2021|
|     3|     Rahul Verma|      Sales| 58000|     2020|
|     4|      Sneha Iyer|    Finance| 81000|     2018|
|     5|     Karan Patel|Engineering| 95000|     2017|
|     6|      Divya Nair|  Marketing| 52000|     2022|
|     7|     Amit Sharma|         HR| 43000|     2021|
|     8|      Neha Gupta|Engineering| 88000|     2018|
|     9|     Rohan Joshi|      Sales| 61000|     2019|
|    10|    Ananya Singh|    Finance| 76000|     2020|
|    11|      Vikram Rao|Engineering|102000|     2016|
|    12|     Pooja Desai|  Marketing| 49000|     2022|
|    13| Siddharth Kumar| Operations| 55000|     2021|
|    14|   Meera Nambiar|         HR| 47000|     2020|
|    15|    Arjun Pillai|Engineering| 91000|     2017|
|    16|  

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('read').getOrCreate()

df = (
    spark.read
         .option('header', 'true')
         .option('inferSchema', 'true')   # fixed typo
         .csv('data/question_01.csv')
)

# Or cleaner as one chain:
df.groupBy('dept').count().orderBy('count', ascending=False).show()

''' 
#or 
df1 = df.groupBy('dept').count().alias('count')
df1.sort(df1['count'].desc()).show()

'''

'''
#way 3
# 1. sort() with .desc()
df1.sort(df1['count'].desc())

# 2. orderBy() with col()
from pyspark.sql.functions import col
df1.orderBy(col('count').desc())

# 3. orderBy() with string + ascending flag
df1.orderBy('count', ascending=False)

'''


+-----------+-----+
|       dept|count|
+-----------+-----+
|Engineering|   13|
|      Sales|    8|
|    Finance|    8|
|         HR|    7|
|  Marketing|    7|
| Operations|    7|
+-----------+-----+



" \n#or \ndf1 = df.groupBy('dept').count().alias('count')\ndf1.sort(df1['count'].desc()).show()\n\n"

```text
groupBy() is a transformation — lazy, no job runs. .show() at the end is the action that triggers execution. Spark builds the full plan first, then optimizes it before running.
```

Q4. Convert the DataFrame to an RDD. Use map() to create a tuple (name, salary * 1.1) for a 10% raise.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('spark').getOrCreate()

csv_file = (
            spark.read\
                    .option('header','true')
                    .option('inferSchema','true')
                    .csv('data/question_01.csv')
)

rdd = csv_file.rdd
print(type(rdd))
# <class 'pyspark.rdd.RDD'>
print(rdd.take(1))
# [Row(emp_id=1, name='Aarav Shah', dept='Engineering', salary=72000, join_year=2019)]

raised = rdd.map(lambda row : (row.name, row.salary * 1.1))
raised.take(5)    # returns a Python list of 5 tuples

'''
# Trigger and print first 5
for name, salary in raised.take(5):
    print(f"{name}: ${salary:,.0f} \n")

    '''

print('.....................')
raised.collect()  # returns ALL rows — careful with large data!


''' 
lambda row: (row.name, row.salary * 1.1)       # attribute style
lambda row: (row['name'], row['salary'] * 1.1)  # dict style
lambda row: (row[1], row[3] * 1.1)              # index style (fragile, avoid)
'''

<class 'pyspark.core.rdd.RDD'>
[Row(emp_id=1, name='Aarav Shah', dept='Engineering', salary=72000, join_year=2019)]
.....................


" \nlambda row: (row.name, row.salary * 1.1)       # attribute style\nlambda row: (row['name'], row['salary'] * 1.1)  # dict style\nlambda row: (row[1], row[3] * 1.1)              # index style (fragile, avoid)\n"

Why does this matter in interviews?
Interviewers ask RDD questions to check if you understand what DataFrames are built on. The honest answer is: in modern PySpark (3.x) you almost never use RDDs directly — but knowing the layer underneath shows depth.

DataFrame = RDD + Schema + Catalyst Optimizer. RDDs give you raw control; DataFrames give you performance.

```text
DataFrames are built ON TOP of RDDs. When Spark optimizes a DataFrame query, it compiles it down to RDD operations. Knowing RDDs shows you understand what's happening under the hood — most candidates skip this.
```

Q5. Write the final DataFrame to a Parquet file, partitioned by dept.

In [2]:

import os
import sys
os.environ['HADOOP_HOME'] = 'C:\\hadoop'


os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("EmployeeData") \
                    .master("local[1]") \
                    .getOrCreate()

# Define the data
data = [
    (1, 'Aarav Shah', 'Engineering', 72000, 2019),
    (2, 'Priya Menon', 'HR', 45000, 2021),
    (3, 'Rahul Verma', 'Sales', 58000, 2020),
    (4, 'Sneha Iyer', 'Finance', 81000, 2018),
    (5, 'Karan Patel', 'Engineering', 95000, 2017),
    (6, 'Divya Nair', 'Marketing', 52000, 2022),
    (7, 'Amit Sharma', 'HR', 43000, 2021),
    (8, 'Neha Gupta', 'Engineering', 88000, 2018),
    (9, 'Rohan Joshi', 'Sales', 61000, 2019),
    (10, 'Ananya Singh', 'Finance', 76000, 2020),
    (11, 'Vikram Rao', 'Engineering', 102000, 2016),
    (12, 'Pooja Desai', 'Marketing', 49000, 2022),
    (13, 'Siddharth Kumar', 'Operations', 55000, 2021),
    (14, 'Meera Nambiar', 'HR', 47000, 2020),
    (15, 'Arjun Pillai', 'Engineering', 91000, 2017),
    (16, 'Kavitha Rajan', 'Finance', 83000, 2019),
    (17, 'Tejas Malhotra', 'Sales', 63000, 2018),
    (18, 'Lakshmi Suresh', 'Operations', 57000, 2020),
    (19, 'Nikhil Bose', 'Engineering', 78000, 2019),
    (20, 'Aditi Chatterjee', 'Marketing', 53000, 2021)
]

# Define the columns
columns = ["emp_id", "name", "dept", "salary", "join_year"]

# Create the DataFrame
df = spark.createDataFrame(data, columns)

# Show the DataFrame
df.show(truncate=False)



df.write.parquet(
    path="data/employee_data.parquet",
    mode="overwrite",
    partitionBy=["dept"]
)

print("Data successfully written to Parquet format!")

+------+----------------+-----------+------+---------+
|emp_id|name            |dept       |salary|join_year|
+------+----------------+-----------+------+---------+
|1     |Aarav Shah      |Engineering|72000 |2019     |
|2     |Priya Menon     |HR         |45000 |2021     |
|3     |Rahul Verma     |Sales      |58000 |2020     |
|4     |Sneha Iyer      |Finance    |81000 |2018     |
|5     |Karan Patel     |Engineering|95000 |2017     |
|6     |Divya Nair      |Marketing  |52000 |2022     |
|7     |Amit Sharma     |HR         |43000 |2021     |
|8     |Neha Gupta      |Engineering|88000 |2018     |
|9     |Rohan Joshi     |Sales      |61000 |2019     |
|10    |Ananya Singh    |Finance    |76000 |2020     |
|11    |Vikram Rao      |Engineering|102000|2016     |
|12    |Pooja Desai     |Marketing  |49000 |2022     |
|13    |Siddharth Kumar |Operations |55000 |2021     |
|14    |Meera Nambiar   |HR         |47000 |2020     |
|15    |Arjun Pillai    |Engineering|91000 |2017     |
|16    |Ka

Py4JJavaError: An error occurred while calling o81.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2078)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2122)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2078)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2122)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(Fi

In [ ]:
import os
import sys

print("HADOOP_HOME is set to:", os.environ.get('HADOOP_HOME'))
print("Is Hadoop in system path?:", any('hadoop' in p.lower() for p in sys.path + os.environ.get('PATH', '').split(';')))

HADOOP_HOME is set to: C:\\hadoop
Is Hadoop in system path?: True


In [ ]:
import os
os.environ['HADOOP_HOME'] = 'C:\\hadoop'

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('spark').getOrCreate()

csv_file = (
            spark.read\
                    .option('header','true')
                    .option('inferSchema','true')
                    .csv('data/question_01.csv')
)

csv_file.show()


# 1. Keyword arguments (your style — clean and readable)
csv_file.write.parquet(
    path='data/question_01.parquet',
    mode='overwrite',
    partitionBy=['dept']
)

# 2. Chained API (most common in production)
csv_file.write \
    .mode('overwrite') \
    .partitionBy('dept') \
    .parquet('data/question_01.parquet')

+------+----------------+-----------+------+---------+
|emp_id|            name|       dept|salary|join_year|
+------+----------------+-----------+------+---------+
|     1|      Aarav Shah|Engineering| 72000|     2019|
|     2|     Priya Menon|         HR| 45000|     2021|
|     3|     Rahul Verma|      Sales| 58000|     2020|
|     4|      Sneha Iyer|    Finance| 81000|     2018|
|     5|     Karan Patel|Engineering| 95000|     2017|
|     6|      Divya Nair|  Marketing| 52000|     2022|
|     7|     Amit Sharma|         HR| 43000|     2021|
|     8|      Neha Gupta|Engineering| 88000|     2018|
|     9|     Rohan Joshi|      Sales| 61000|     2019|
|    10|    Ananya Singh|    Finance| 76000|     2020|
|    11|      Vikram Rao|Engineering|102000|     2016|
|    12|     Pooja Desai|  Marketing| 49000|     2022|
|    13| Siddharth Kumar| Operations| 55000|     2021|
|    14|   Meera Nambiar|         HR| 47000|     2020|
|    15|    Arjun Pillai|Engineering| 91000|     2017|
|    16|  

Py4JJavaError: An error occurred while calling o699.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2078)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2122)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2078)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2122)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(F

```text

Q1 ✓  SparkSession + read CSV
Q2 ✓  filter + select
Q3 ✓  groupBy + count
Q4 ✓  RDD + map()
Q5 ✓  write Parquet partitioned

```

In [3]:
import os
from pyspark.sql import SparkSession

# 1. Safety net for Windows Hadoop issues (forces Spark to see your Hadoop folder)
os.environ['HADOOP_HOME'] = 'C:\\hadoop'

# 2. Initialize Spark Session
print("Starting Spark Session...")
spark = SparkSession.builder \
    .appName("LocalEmployeeJob") \
    .master("local[*]") \
    .getOrCreate()

# Optional: Set log level to ERROR to hide the massive walls of Spark INFO text
spark.sparkContext.setLogLevel("ERROR")

# 3. Define the data
data = [
    (1, 'Aarav Shah', 'Engineering', 72000, 2019),
    (2, 'Priya Menon', 'HR', 45000, 2021),
    (3, 'Rahul Verma', 'Sales', 58000, 2020),
    (4, 'Sneha Iyer', 'Finance', 81000, 2018),
    (5, 'Karan Patel', 'Engineering', 95000, 2017),
    (6, 'Divya Nair', 'Marketing', 52000, 2022),
    (7, 'Amit Sharma', 'HR', 43000, 2021),
    (8, 'Neha Gupta', 'Engineering', 88000, 2018),
    (9, 'Rohan Joshi', 'Sales', 61000, 2019),
    (10, 'Ananya Singh', 'Finance', 76000, 2020)
]

columns = ["emp_id", "name", "dept", "salary", "join_year"]

# 4. Create and show the DataFrame
print("Creating DataFrame...")
df = spark.createDataFrame(data, columns)
df.show(truncate=False)

# 5. Write to Parquet (Partitioned)
output_path = "data/employee_data.parquet"
print(f"Writing data to Parquet format at: {output_path}")

df.write.parquet(
    path=output_path,
    mode="overwrite",
    partitionBy=["dept"]
)

print("Job completed successfully!")

# 6. Cleanly shut down Spark
spark.stop()

Starting Spark Session...
Creating DataFrame...
+------+------------+-----------+------+---------+
|emp_id|name        |dept       |salary|join_year|
+------+------------+-----------+------+---------+
|1     |Aarav Shah  |Engineering|72000 |2019     |
|2     |Priya Menon |HR         |45000 |2021     |
|3     |Rahul Verma |Sales      |58000 |2020     |
|4     |Sneha Iyer  |Finance    |81000 |2018     |
|5     |Karan Patel |Engineering|95000 |2017     |
|6     |Divya Nair  |Marketing  |52000 |2022     |
|7     |Amit Sharma |HR         |43000 |2021     |
|8     |Neha Gupta  |Engineering|88000 |2018     |
|9     |Rohan Joshi |Sales      |61000 |2019     |
|10    |Ananya Singh|Finance    |76000 |2020     |
+------+------------+-----------+------+---------+

Writing data to Parquet format at: data/employee_data.parquet


Py4JJavaError: An error occurred while calling o110.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2078)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2122)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2078)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2122)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(F